# Khía cạnh được nhắc tới (ước lượng bằng từ khóa)

**Mục đích:** biết sơ bộ người ta bàn về khía cạnh nào nhiều nhất, và bộ 4 máy có đủ bình luận cho từng khía cạnh không, trước khi gán nhãn (tầng 3). Đây là ước lượng thô bằng từ khóa, không phải nhãn. Cập nhật lần cuối: 2026-11-26.

**Nội dung**

1. Đọc file đã xử lý
2. Khía cạnh được nhắc tới: bảng, từ khóa đóng góp, ví dụ
3. Bộ 4 máy × khía cạnh

**Đọc file nào:** `data/processed/comments.jsonl` và manifest của nó, do lệnh tầng 2 ghi ra. Chưa có file thì chạy trước, từ gốc repo: `.\.venv\Scripts\python.exe -m src.processing.run` (hướng dẫn tầng 2 mục 3). Notebook không ghi file nào.

In [ ]:
import json
import re
import unicodedata
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

from helpers import *   # ROOT, đường dẫn dữ liệu, N_SHOW, doc_jsonl, doc_da_xu_ly, doc_bo_may, lo_ra, bao_cao, display, pd
thiet_lap_hien_thi()

## 1. Đọc file đã xử lý

In [ ]:
df, man = doc_da_xu_ly(["comment_id", "model", "text_clean"])
df["la_null"] = df["text_clean"] == ""   # bình luận rỗng sau làm sạch

## 2. Khía cạnh được nhắc tới (ước lượng nhanh bằng từ khóa)

**Để làm gì:** biết sơ bộ người ta bàn về khía cạnh nào nhiều nhất: hiệu năng, pin, camera… Dùng 11 khía cạnh của bộ nhãn chốt (26/11, `src/labeling/final_labels.py`; [thiết kế dữ liệu](../docs/0-chung/thiet-ke-du-lieu.md#danh-mục-mã)):

| Khía cạnh | Gồm |
|---|---|
| Pin & sạc | dung lượng, thời lượng dùng, tốc độ sạc |
| Màn hình | độ sáng, tần số quét, độ phân giải, tấm nền |
| Hiệu năng | chip, độ mượt, chơi game, RAM, nóng máy |
| Camera | chụp, quay, zoom, selfie |
| Thiết kế | ngoại hình, chất liệu, cầm nắm, màu máy |
| Độ bền | độ bền, rơi vỡ, chống nước |
| Phần mềm | hệ điều hành, giao diện, cập nhật, tính năng, AI |
| Phần cứng khác | loa, rung, vân tay, Face ID, sóng, bộ nhớ, bút |
| Giá | giá bán, đáng tiền, giữ giá, khuyến mãi |
| Mua bán | chính hãng/xách tay, cửa hàng, trả góp, lên đời, định mua |
| Bảo hành & sửa chữa | bảo hành, sửa chữa, đổi trả, dịch vụ |

Từ khóa không đoán `may_chung` (cả máy) và `hang_nguoi_dung` (hãng, người dùng).

**Cách làm:**
- Mỗi khía cạnh có một danh sách từ khóa. Bình luận chứa từ khóa nào thì tính là có nhắc khía cạnh đó. Bộ từ khóa nằm ở `src/models/aspect/keywords.py`, dùng chung với luật mốc của bước suy luận (`docs/4-mo-hinh/dac-ta-suy-luan.md`): sửa từ khóa ở đó, không sửa trong notebook.
- Một bình luận có thể nhắc nhiều khía cạnh, nên cộng các % lại sẽ lớn hơn 100%.
- Chạy trên `text_clean` của Level 1, bỏ bình luận rỗng; không phân biệt chữ hoa, chữ thường.
- Các từ dễ bắt nhầm đã có luật loại trừ:
  - "cam" trong "màu cam", "cam on" (cảm ơn không dấu);
  - "trâu" trong "trẻ trâu";
  - "rung" trong "chống rung";
  - "sóng" trong "lên sóng";
  - "ai" thường (nghĩa là "người nào"): chỉ tính khi viết hoa "AI", hoặc "Galaxy AI".

**Giới hạn — đọc trước khi dùng số:**
- Đây là **ước lượng thô**, không phải nhãn.
- Từ khóa bỏ sót cách nói không có từ khóa, vd "máy đẹp" không được tính vào Thiết kế, vì "đẹp" dùng cho mọi thứ.
- Từ khóa cũng có thể bắt nhầm. Ô cuối mục in ví dụ ngẫu nhiên để kiểm bằng mắt.
- Chưa phân biệt khen hay chê, chưa biết bình luận nói về máy nào. Bình luận spam, minigame vẫn được tính.

In [ ]:
from src.models.aspect.keywords import TEN_KHIA_CANH, TU_KHOA as TU_KHOA_THEO_MA   # dùng chung với luật mốc của bước suy luận
TU_KHOA = {TEN_KHIA_CANH[ma]: ds for ma, ds in TU_KHOA_THEO_MA.items()}   # khía cạnh -> [(nhãn từ khóa, regex)]

van_ban = df.loc[~df["la_null"], "text_clean"]   # bình luận có nội dung, đã qua Level 1
N = len(van_ban)
trung_tu = {}                                    # (khía cạnh, từ khóa) -> cột đúng/sai
for kc, ds in TU_KHOA.items():
    for nhan, mau in ds:
        trung_tu[(kc, nhan)] = van_ban.str.contains(re.compile(mau, re.IGNORECASE), regex=True)
trung_tu = pd.DataFrame(trung_tu)
nhac = trung_tu.T.groupby(level=0, sort=False).any().T    # khía cạnh -> có nhắc hay không

so_kc = nhac.sum(axis=1)
bang_kc = pd.DataFrame({"số bình luận": nhac.sum(), "% bình luận": (100 * nhac.mean()).round(2)})
bang_kc.loc["(không khớp khía cạnh nào)"] = [int((so_kc == 0).sum()), round(100 * (so_kc == 0).mean(), 2)]
bang_kc["số bình luận"] = bang_kc["số bình luận"].astype(int)
print(f"Tính trên {N:,} bình luận có nội dung")
display(bang_kc.sort_values("số bình luận", ascending=False))

phan_bo = so_kc.clip(upper=3).value_counts().sort_index()
phan_bo.index = ["0 khía cạnh", "1 khía cạnh", "2 khía cạnh", "3 khía cạnh trở lên"][:len(phan_bo)]
display(pd.DataFrame({"số bình luận": phan_bo, "% bình luận": (100 * phan_bo / N).round(2)}))

ve = bang_kc.drop("(không khớp khía cạnh nào)")["% bình luận"].sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(ve.index, ve.values, color="#4472c4")
for y, v in enumerate(ve.values):
    ax.text(v + 0.2, y, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlabel("% bình luận nhắc tới (ước lượng bằng từ khóa)")
ax.set_title("Khía cạnh được nhắc tới")
plt.show()

### Từ khóa nào đóng góp nhiều nhất

**Để làm gì:** xem mỗi khía cạnh được đếm chủ yếu nhờ từ khóa nào. Một từ khóa chiếm gần hết một khía cạnh thì cần xem nó có bắt nhầm không. Vd nếu "iOS, Android" chiếm phần lớn Phần mềm, thì nhiều bình luận đó có thể chỉ là cãi nhau iPhone với Android.

**Ô code làm gì:** với từng khía cạnh, in số bình luận khớp mỗi từ khóa và % trên tổng số bình luận. Một bình luận có thể khớp nhiều từ khóa.

In [ ]:
bang_tu = pd.DataFrame({"số bình luận": trung_tu.sum(), "% bình luận": (100 * trung_tu.mean()).round(2)})
bang_tu.index.names = ["khía cạnh", "từ khóa"]
display(bang_tu.sort_values(["khía cạnh", "số bình luận"], ascending=[True, False]))

### Ví dụ ngẫu nhiên từng khía cạnh

**Để làm gì:** kiểm bằng mắt xem từ khóa bắt đúng hay nhầm.

**Ô code làm gì:**
- In `SO_VI_DU` bình luận ngẫu nhiên cho mỗi khía cạnh, cắt ở 200 ký tự.
- Cột `từ khóa khớp` cho biết bình luận được tính nhờ từ nào.
- Đổi `random_state` để xem bộ ví dụ khác.

**Đọc kết quả:** nếu một khía cạnh có nhiều ví dụ bắt nhầm, thì % của nó ở bảng đầu đang bị thổi phồng.

In [ ]:
SO_VI_DU = 8
for kc in TU_KHOA:
    co = nhac.index[nhac[kc]]
    mau = pd.Series(co).sample(min(SO_VI_DU, len(co)), random_state=0)
    print(f"\n{kc}: {len(co):,} bình luận")
    display(pd.DataFrame({
        "text_clean": van_ban.loc[mau].str.replace("\n", " ⏎ ", regex=False).str.slice(0, 200).values,
        "từ khóa khớp": [", ".join(n for (k, n) in trung_tu.columns if k == kc and trung_tu.at[i, (k, n)]) for i in mau],
    }))

## 3. Bộ 4 máy × khía cạnh

**Để làm gì:** thử bộ 4 máy (X, đời trước P, hai đối thủ R1, R2) trước khi chốt ở Công việc 2: mỗi máy có đủ bình luận cho từng khía cạnh không.

**Ô code làm gì:**
- Đếm số bình luận **nhắc** từng khía cạnh (từ khóa của mục 2), dưới video của từng máy. Video so sánh `a|b` tính cho cả hai máy.
- Ô nào dưới `NGUONG_O` = 30 được đánh ⚠. `muc-tieu.md` quy định ô có khen + chê dưới 30 là "ít dữ liệu". Số khen + chê luôn ít hơn số nhắc, nên ô ⚠ **chắc chắn** thiếu dữ liệu; ô không ⚠ thì chưa chắc đủ.

**Bộ máy lấy từ `config/analysis.yaml`**, cùng một bộ với phân tích Q1–Q5. File đó hiện vẫn là bộ cũ, chờ Công việc 2 chốt. Muốn thử bộ khác thì gán lại `BO_MAY`, vd `{"X": "s26u", "P": "s25u", "R1": "ip17prm", "R2": "xiaomi17u"}`. Số bình luận theo tuần của bộ máy: notebook `1_raw_audit.ipynb` mục 3.4.

**Lưu ý:** đây là máy **của video**, chưa phải máy được nhắc trong bình luận (Công việc 10).

In [ ]:
BO_MAY = doc_bo_may()   # từ config/analysis.yaml
NGUONG_O = 30

nhan = [f"{vai} = {m}" for vai, m in BO_MAY.items()]
nhac_may = nhac.assign(may=df.loc[nhac.index, "model"].fillna("").str.split("|")).explode("may")
o = (nhac_may[nhac_may["may"].isin(BO_MAY.values())].groupby("may")[list(TU_KHOA)].sum()
       .reindex(list(BO_MAY.values()), fill_value=0).astype(int))
hien = o.apply(lambda cot: cot.map(lambda v: f"⚠ {v:,}" if v < NGUONG_O else f"{v:,}"))
hien.index = nhan
so_o_thieu = int((o < NGUONG_O).sum().sum())
print(f"Số bình luận nhắc từng khía cạnh. ⚠ = dưới {NGUONG_O}: chắc chắn thiếu. Ô thiếu: {so_o_thieu} / {o.size}")
display(hien)